## Loading and Inspecting the Data

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
filename = 'Call-Center-Dataset.xlsx'
src_path = Path.cwd().parent / "data/raw" / filename
src_path

PosixPath('/home/ashish_ubuntu/da_projects/call-center-staffing-analytics/data/raw/Call-Center-Dataset.xlsx')

In [3]:
df = pd.read_excel(src_path)
df.head()

,Call Id,Date,Agent,Department,Answered (Y/N),Resolved,Speed of Answer,AvgTalkDuration,Satisfaction rating
0,ID0001,2015-01-01 09:12:58.000,Diane,Washing Machine,Y,Y,109.0,00:02:23.210000,3.0
1,ID0002,2015-01-01 09:12:57.600,Becky,Air Conditioner,Y,N,70.0,00:04:02.323000,3.0
2,ID0003,2015-01-01 09:47:31.200,Stewart,Washing Machine,Y,Y,10.0,00:02:11.202000,3.0
3,ID0004,2015-01-01 09:47:31.200,Greg,Washing Machine,Y,Y,53.0,00:00:37.481000,2.0
4,ID0005,2015-01-01 10:00:28.800,Becky,Toaster,Y,Y,95.0,00:01:00.083000,3.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Call Id              5000 non-null   str           
 1   Date                 5000 non-null   datetime64[us]
 2   Agent                5000 non-null   str           
 3   Department           5000 non-null   str           
 4   Answered (Y/N)       5000 non-null   str           
 5   Resolved             5000 non-null   str           
 6   Speed of Answer      4054 non-null   float64       
 7   AvgTalkDuration      4054 non-null   object        
 8   Satisfaction rating  4054 non-null   float64       
dtypes: datetime64[us](1), float64(2), object(1), str(5)
memory usage: 351.7+ KB


In [5]:
df.describe()

,Date,Speed of Answer,Satisfaction rating
count,5000,4054.000000,4054.000000
mean,2015-02-13 19:04:05.491279,67.520720,3.403552
min,2015-01-01 09:12:57.600000,10.000000,1.000000
25%,2015-01-23 11:36:36,39.000000,3.000000
50%,2015-02-13 11:52:48,68.000000,3.000000
75%,2015-03-06 17:22:33.600000,97.000000,4.000000
max,2015-03-31 17:39:50.400000,125.000000,5.000000
std,NaN,33.592872,1.212220


In [6]:
df.isna().sum()

Call Id                  0
Date                     0
Agent                    0
Department               0
Answered (Y/N)           0
Resolved                 0
Speed of Answer        946
AvgTalkDuration        946
Satisfaction rating    946
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
unique_values = {col: df[col].nunique() for col in df}
unique_values

{'Call Id': 5000,
 'Date': 2422,
 'Agent': 8,
 'Department': 5,
 'Answered (Y/N)': 2,
 'Resolved': 2,
 'Speed of Answer': 116,
 'AvgTalkDuration': 4038,
 'Satisfaction rating': 5}

In [9]:
df.loc[df['AvgTalkDuration'].isna()].head()

,Call Id,Date,Agent,Department,Answered (Y/N),Resolved,Speed of Answer,AvgTalkDuration,Satisfaction rating
5,ID0006,2015-01-01 10:00:28.800,Stewart,Air Conditioner,N,N,NaN,NaN,NaN
10,ID0011,2015-01-01 11:15:21.600,Joe,Toaster,N,N,NaN,NaN,NaN
17,ID0018,2015-01-01 11:57:07.200,Becky,Fridge,N,N,NaN,NaN,NaN
18,ID0019,2015-01-01 12:01:26.400,Jim,Television,N,N,NaN,NaN,NaN
22,ID0023,2015-01-01 12:02:52.800,Martha,Air Conditioner,N,N,NaN,NaN,NaN


In [10]:
df.loc[df['AvgTalkDuration'].isna(), 'Answered (Y/N)'].unique()

<StringArray>
['N']
Length: 1, dtype: str

##### Observations:

1. _Date_ column should be split into date and time columns respectively, with time column rounding for the nearest whole second
2. _AvgTalkDuration_ needs to be converted to a numeric column of seconds
3. Blank rows in _Speed of Answer_, _AvgTalkDuration_ and _Satisfaction rating_ appears to be limited to unanswered calls, blank values can left alone
4. Column headers should be standardized by changing the case to lowercase and using snake_case convention
5. _Answered (Y/N)_ and _Resolved_ column flags should be standardized as "Yes" and "No" respectively

## Cleaning the data

#### Normalizing the column headers

Ensuring the column headers follow lowercase, snake_case standards, while removing unnecessary characters such as parantheses

In [11]:
cleaned_df = df.rename(columns={'Answered (Y/N)': 'answered', 'AvgTalkDuration': 'avg_talk_duration'})
cleaned_df.columns = (
    cleaned_df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'\s+', '_', regex=True)
)
cleaned_df.head()

,call_id,date,agent,department,answered,resolved,speed_of_answer,avg_talk_duration,satisfaction_rating
0,ID0001,2015-01-01 09:12:58.000,Diane,Washing Machine,Y,Y,109.0,00:02:23.210000,3.0
1,ID0002,2015-01-01 09:12:57.600,Becky,Air Conditioner,Y,N,70.0,00:04:02.323000,3.0
2,ID0003,2015-01-01 09:47:31.200,Stewart,Washing Machine,Y,Y,10.0,00:02:11.202000,3.0
3,ID0004,2015-01-01 09:47:31.200,Greg,Washing Machine,Y,Y,53.0,00:00:37.481000,2.0
4,ID0005,2015-01-01 10:00:28.800,Becky,Toaster,Y,Y,95.0,00:01:00.083000,3.0


### Splitting the timestamp column into date and time columns respectively

Splitting the timestamp into date and time columns will help with aggregating the data on a date/month/year granularity, while the time column helps with time series analysis if required.

The raw data includes the timestamp at a micro-second precision level, however that level of precision is irrelevant to a day-wise analysis. As such, I'm choosing to round it to the nearest second.

In [12]:
cleaned_df = cleaned_df.assign(
    time=lambda df: df['date'].dt.round('s').dt.time,
    date=lambda df: df['date'].dt.date,
)
cleaned_df.head()

,call_id,date,agent,department,answered,resolved,speed_of_answer,avg_talk_duration,satisfaction_rating,time
0,ID0001,2015-01-01,Diane,Washing Machine,Y,Y,109.0,00:02:23.210000,3.0,09:12:58
1,ID0002,2015-01-01,Becky,Air Conditioner,Y,N,70.0,00:04:02.323000,3.0,09:12:58
2,ID0003,2015-01-01,Stewart,Washing Machine,Y,Y,10.0,00:02:11.202000,3.0,09:47:31
3,ID0004,2015-01-01,Greg,Washing Machine,Y,Y,53.0,00:00:37.481000,2.0,09:47:31
4,ID0005,2015-01-01,Becky,Toaster,Y,Y,95.0,00:01:00.083000,3.0,10:00:29


### Converting the *avg_talk_duration* column into a numeric seconds column

*avg_talk_duration* is the duration of each call, converting it into a whole number representing seconds elapsed will help with arithmetic and statistical calculations on the column during analysis.

In [13]:
def convert_time_to_seconds(time_obj):
    if pd.isnull(time_obj):
        return float('nan')
    hour = int(time_obj.hour)
    mins = int(time_obj.minute)
    secs = int(time_obj.second)
    return float((hour * 3600) + (mins * 60) + secs)

cleaned_df['avg_talk_duration'] = cleaned_df['avg_talk_duration'].apply(convert_time_to_seconds)
cleaned_df.head()

,call_id,date,agent,department,answered,resolved,speed_of_answer,avg_talk_duration,satisfaction_rating,time
0,ID0001,2015-01-01,Diane,Washing Machine,Y,Y,109.0,143.0,3.0,09:12:58
1,ID0002,2015-01-01,Becky,Air Conditioner,Y,N,70.0,242.0,3.0,09:12:58
2,ID0003,2015-01-01,Stewart,Washing Machine,Y,Y,10.0,131.0,3.0,09:47:31
3,ID0004,2015-01-01,Greg,Washing Machine,Y,Y,53.0,37.0,2.0,09:47:31
4,ID0005,2015-01-01,Becky,Toaster,Y,Y,95.0,60.0,3.0,10:00:29


### Standardizing _answered_ and _resolved_ flags

The _answered_ and _resolved_ columns in the raw data comes with a 'Y' and 'N' flag for 'Yes' and 'No' respectively, I'm choosing to standardize those flags to lowercase 'yes' and 'no' respectively to ensure uniform formatting.

In [14]:
flag_map = {'Y': 'yes', 'N': 'no', 'y': 'yes', 'n': 'no'}

In [15]:
cleaned_df = cleaned_df.assign(
    answered=lambda df: df['answered'].map(flag_map, na_action='ignore'),
    resolved=lambda df: df['resolved'].map(flag_map, na_action='ignore'),
)
cleaned_df.head()

,call_id,date,agent,department,answered,resolved,speed_of_answer,avg_talk_duration,satisfaction_rating,time
0,ID0001,2015-01-01,Diane,Washing Machine,yes,yes,109.0,143.0,3.0,09:12:58
1,ID0002,2015-01-01,Becky,Air Conditioner,yes,no,70.0,242.0,3.0,09:12:58
2,ID0003,2015-01-01,Stewart,Washing Machine,yes,yes,10.0,131.0,3.0,09:47:31
3,ID0004,2015-01-01,Greg,Washing Machine,yes,yes,53.0,37.0,2.0,09:47:31
4,ID0005,2015-01-01,Becky,Toaster,yes,yes,95.0,60.0,3.0,10:00:29


### Cleaning the strings on the remaining columns

Trimming whitespaces and standardizing case in the remaining 3 string columns for consistency. 

I chose to use *.str.capitalize()* on the Agent column and *.str.title()* on the Department column intentionally, as agent is single word column while department is multi word columns and *.str.capitalize()* would not give consistent results.

In [16]:
cleaned_df = cleaned_df.assign(
    call_id=lambda df: df['call_id'].str.strip(),
    agent=lambda df: df['agent'].str.strip().str.capitalize(),
    department=lambda df: df['department'].str.strip().str.title()
)

cleaned_df.head()

,call_id,date,agent,department,answered,resolved,speed_of_answer,avg_talk_duration,satisfaction_rating,time
0,ID0001,2015-01-01,Diane,Washing Machine,yes,yes,109.0,143.0,3.0,09:12:58
1,ID0002,2015-01-01,Becky,Air Conditioner,yes,no,70.0,242.0,3.0,09:12:58
2,ID0003,2015-01-01,Stewart,Washing Machine,yes,yes,10.0,131.0,3.0,09:47:31
3,ID0004,2015-01-01,Greg,Washing Machine,yes,yes,53.0,37.0,2.0,09:47:31
4,ID0005,2015-01-01,Becky,Toaster,yes,yes,95.0,60.0,3.0,10:00:29


### Rearranging the columns to align it with raw data ordering

In [17]:
cleaned_df = cleaned_df[
    [
        'call_id', 
        'date', 
        'time', 
        'agent', 
        'department', 
        'answered', 
        'resolved', 
        'speed_of_answer', 
        'avg_talk_duration', 
        'satisfaction_rating'
    ]
]

## Result

In [18]:
cleaned_df.head()

,call_id,date,time,agent,department,answered,resolved,speed_of_answer,avg_talk_duration,satisfaction_rating
0,ID0001,2015-01-01,09:12:58,Diane,Washing Machine,yes,yes,109.0,143.0,3.0
1,ID0002,2015-01-01,09:12:58,Becky,Air Conditioner,yes,no,70.0,242.0,3.0
2,ID0003,2015-01-01,09:47:31,Stewart,Washing Machine,yes,yes,10.0,131.0,3.0
3,ID0004,2015-01-01,09:47:31,Greg,Washing Machine,yes,yes,53.0,37.0,2.0
4,ID0005,2015-01-01,10:00:29,Becky,Toaster,yes,yes,95.0,60.0,3.0


In [19]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   call_id              5000 non-null   str    
 1   date                 5000 non-null   object 
 2   time                 5000 non-null   object 
 3   agent                5000 non-null   str    
 4   department           5000 non-null   str    
 5   answered             5000 non-null   str    
 6   resolved             5000 non-null   str    
 7   speed_of_answer      4054 non-null   float64
 8   avg_talk_duration    4054 non-null   float64
 9   satisfaction_rating  4054 non-null   float64
dtypes: float64(3), object(2), str(5)
memory usage: 390.8+ KB


In [20]:
cleaned_df.describe()

,speed_of_answer,avg_talk_duration,satisfaction_rating
count,4054.000000,4054.000000,4054.000000
mean,67.520720,224.424026,3.403552
std,33.592872,111.371053,1.212220
min,10.000000,30.000000,1.000000
25%,39.000000,130.000000,3.000000
50%,68.000000,226.000000,3.000000
75%,97.000000,319.000000,4.000000
max,125.000000,419.000000,5.000000


In [21]:
cleaned_df.isna().sum()

call_id                  0
date                     0
time                     0
agent                    0
department               0
answered                 0
resolved                 0
speed_of_answer        946
avg_talk_duration      946
satisfaction_rating    946
dtype: int64

In [22]:
cleaned_df.duplicated().sum()

np.int64(0)

In [23]:
unique_values = {col: cleaned_df[col].nunique() for col in cleaned_df}
unique_values

{'call_id': 5000,
 'date': 90,
 'time': 375,
 'agent': 8,
 'department': 5,
 'answered': 2,
 'resolved': 2,
 'speed_of_answer': 116,
 'avg_talk_duration': 390,
 'satisfaction_rating': 5}

## Export

In [24]:
output_filename = 'processed_data.csv'
output_dir = Path.cwd().parent / "data/processed"
output_dir.mkdir(exist_ok=True, parents=True)
output_path = output_dir / output_filename

cleaned_df.to_csv(output_path, index=False)
